# Spot The Mask Challenge
**Objective**: Predict the likelihood that an image contains at least one person wearing a mask.
**Metric**: Log Loss

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Path Configuration

In [ ]:
def setup_paths():
    base_dir = '/kaggle/input'
    train_csv = 'train_labels.csv'
    img_dir = 'images'
    
    if os.path.exists(base_dir):
        for root, dirs, files in os.walk(base_dir):
            if 'train_labels.csv' in files:
                train_csv = os.path.join(root, 'train_labels.csv')
            if 'images' in dirs:
                img_dir = os.path.join(root, 'images')
                
    return train_csv, img_dir

TRAIN_CSV, IMG_DIR = setup_paths()
print(f"CSV Path: {TRAIN_CSV}")
print(f"Image Directory: {IMG_DIR}")

## 2. Dataset Class

In [ ]:
class MaskDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['image_id']
        img_path = os.path.join(self.img_dir, img_name)
        
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        label = self.df.iloc[idx]['target']
        return image, torch.tensor(label, dtype=torch.float32)

## 3. Model (EfficientNet-B0)

In [ ]:
def get_model():
    model = models.efficientnet_b0(pretrained=True)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    return model.to(device)

## 4. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        # Validation
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                val_preds.extend(outputs.cpu().numpy())
                val_labels.extend(labels.numpy())
        
        loss_val = log_loss(val_labels, val_preds)
        print(f"Epoch {epoch+1}, Train Loss: {train_loss/len(train_loader):.4f}, Val LogLoss: {loss_val:.4f}")

## 5. Main Execution

In [ ]:
if os.path.exists(TRAIN_CSV) and os.path.exists(IMG_DIR):
    df = pd.read_csv(TRAIN_CSV)
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
    
    transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_ds = MaskDataset(train_df, IMG_DIR, transform)
    val_ds = MaskDataset(val_df, IMG_DIR, transform)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
    
    model = get_model()
    train_model(model, train_loader, val_loader)
else:
    print("Dataset not found. Please attach the 'Spot the Mask' dataset on Kaggle.")